# Step 3.1.2.2: Shape Graphs批量处理与结果保存

## 目标

1. 加载Shape Graph数据
2. 批量处理所有帧进行阵型识别
3. 应用时序平滑
4. 保存结果

## 前置条件

需要先运行3.1.2.1定义核心函数

## 1. 导入库和配置

In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# 比赛信息
GAME_ID = 10517  # 2022世界杯决赛
HOME_TEAM_ID = '364'  # 阿根廷

# 数据路径
DATA_DIR = Path('../../../data/morph_test')
GRAPHS_DIR = DATA_DIR / 'shape_graphs' / 'graphs'
OUTPUT_DIR = DATA_DIR / 'shapegraphs_baseline'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ 配置完成")
print(f"比赛: 2022世界杯决赛 (Game {GAME_ID})")
print(f"分析球队: 阿根廷 (Team ID: {HOME_TEAM_ID})")
print(f"Shape Graph目录: {GRAPHS_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

✅ 配置完成
比赛: 2022世界杯决赛 (Game 10517)
分析球队: 阿根廷 (Team ID: 364)
Shape Graph目录: ..\..\..\data\morph_test\shape_graphs\graphs
输出目录: ..\..\..\data\morph_test\shapegraphs_baseline


## 2. 导入核心函数

从3.1.2.1复制核心函数定义（%run无法运行.ipynb文件）

In [2]:
from scipy.spatial import ConvexHull

def compute_convex_hull_split(positions):
    if len(positions) < 3:
        return np.median(positions[:, 0])
    try:
        hull = ConvexHull(positions)
        hull_points = positions[hull.vertices]
        split_x = np.median(hull_points[:, 0])
        return split_x
    except:
        return np.median(positions[:, 0])

def assign_vertical_levels(positions, graph):
    N = len(positions)
    levels = np.zeros(N, dtype=int)
    x_coords = positions[:, 0]
    sorted_indices = np.argsort(x_coords)
    split1 = compute_convex_hull_split(positions)
    forward_mask = x_coords > split1
    backward_mask = ~forward_mask
    
    if np.sum(forward_mask) > 0:
        forward_positions = positions[forward_mask]
        forward_indices = np.where(forward_mask)[0]
        if len(forward_positions) > 1:
            split_forward = compute_convex_hull_split(forward_positions)
            for i, idx in enumerate(forward_indices):
                if positions[idx, 0] > split_forward:
                    levels[idx] = 4
                else:
                    levels[idx] = 3
        else:
            levels[forward_indices] = 4
    
    if np.sum(backward_mask) > 0:
        backward_positions = positions[backward_mask]
        backward_indices = np.where(backward_mask)[0]
        if len(backward_positions) > 2:
            split_backward = compute_convex_hull_split(backward_positions)
            for i, idx in enumerate(backward_indices):
                if positions[idx, 0] < split_backward:
                    levels[idx] = 0
                else:
                    mid_mask = (positions[backward_indices, 0] >= split_backward)
                    if np.sum(mid_mask) > 1:
                        mid_positions = positions[backward_indices[mid_mask]]
                        split_mid = np.median(mid_positions[:, 0])
                        if positions[idx, 0] > split_mid:
                            levels[idx] = 2
                        else:
                            levels[idx] = 1
                    else:
                        levels[idx] = 2
        else:
            for i, idx in enumerate(backward_indices):
                if i < len(backward_indices) // 2:
                    levels[idx] = 0
                else:
                    levels[idx] = 1
    return levels

def assign_horizontal_positions(positions, levels):
    N = len(positions)
    h_positions = np.zeros(N, dtype=int)
    for level in range(5):
        level_mask = (levels == level)
        if np.sum(level_mask) == 0:
            continue
        level_indices = np.where(level_mask)[0]
        level_y = positions[level_mask, 1]
        sorted_y_indices = np.argsort(level_y)
        n_players = len(sorted_y_indices)
        if n_players == 1:
            h_positions[level_indices[sorted_y_indices[0]]] = 2
        elif n_players == 2:
            h_positions[level_indices[sorted_y_indices[0]]] = 0
            h_positions[level_indices[sorted_y_indices[1]]] = 4
        elif n_players == 3:
            h_positions[level_indices[sorted_y_indices[0]]] = 0
            h_positions[level_indices[sorted_y_indices[1]]] = 2
            h_positions[level_indices[sorted_y_indices[2]]] = 4
        elif n_players == 4:
            h_positions[level_indices[sorted_y_indices[0]]] = 0
            h_positions[level_indices[sorted_y_indices[1]]] = 1
            h_positions[level_indices[sorted_y_indices[2]]] = 3
            h_positions[level_indices[sorted_y_indices[3]]] = 4
        else:
            for i, idx in enumerate(sorted_y_indices):
                h_pos = int(i * 4 / (n_players - 1))
                h_positions[level_indices[idx]] = h_pos
    return h_positions

def identify_shape_graph_edge_players(graph, positions, levels):
    """识别Shape Graph边上的球员"""
    N = len(positions)
    on_edge_mask = np.zeros(N, dtype=bool)
    
    if graph is None or graph.number_of_nodes() == 0:
        return on_edge_mask
    
    try:
        hull = ConvexHull(positions)
        hull_vertices = set(hull.vertices)
        
        for edge in graph.edges():
            u, v = edge
            if u in hull_vertices and v in hull_vertices:
                on_edge_mask[u] = True
                on_edge_mask[v] = True
        
        return on_edge_mask
    except:
        min_x = np.min(positions[:, 0])
        threshold = min_x + 5.0
        on_edge_mask = positions[:, 0] <= threshold
        return on_edge_mask

def infer_formation_from_levels(levels, graph=None, positions=None):
    """
    根据垂直层级和Shape Graph边推断阵型
    
    Code格式：b(dma)f
    返回三种格式：
    - formation_simple: 简化格式（如'4-3-3'）
    - formation_code: Code格式（如'4(003)3'）
    - formation_detailed: 5位数字（如'40033'）
    """
    level_counts = np.bincount(levels, minlength=5)
    n_B = level_counts[0]
    n_DM = level_counts[1]
    n_M = level_counts[2]
    n_AM = level_counts[3]
    n_F = level_counts[4]
    
    if graph is not None and positions is not None:
        on_edge_mask = identify_shape_graph_edge_players(graph, positions, levels)
        dm_indices = np.where(levels == 1)[0]
        n_DM_on_edge = np.sum(on_edge_mask[dm_indices])
        n_DM_off_edge = n_DM - n_DM_on_edge
        b = n_B + n_DM_on_edge
        d = n_DM_off_edge
    else:
        b = n_B
        d = n_DM
    
    m = n_M
    a = n_AM
    f = n_F
    
    formation_code = f"{b}({d}{m}{a}){f}"
    formation_detailed = f"{b}{d}{m}{a}{f}"
    formation_simple = f"{b}-{d+m+a}-{f}"
    
    return formation_simple, formation_code, formation_detailed

print("✅ 核心函数定义完成（新增Shape Graph边识别）")

✅ 核心函数定义完成（新增Shape Graph边识别）


## 3. 加载Shape Graph数据

In [3]:
# 加载Shape Graph文件列表
graph_files = sorted(GRAPHS_DIR.glob('shape_graph_*.pkl'))

print(f"找到 {len(graph_files)} 个Shape Graph文件")
if len(graph_files) > 0:
    print(f"第一个文件: {graph_files[0].name}")
    print(f"最后一个文件: {graph_files[-1].name}")

找到 50084 个Shape Graph文件
第一个文件: shape_graph_10018.pkl
最后一个文件: shape_graph_98893.pkl


## 4. 批量处理：阵型识别

In [4]:
# 批量处理所有帧
results = []

print("=== 开始批量处理 ===")
print(f"处理 {len(graph_files)} 个Shape Graph文件...\n")

for graph_file in tqdm(graph_files, desc="识别阵型"):
    # 加载Shape Graph数据
    with open(graph_file, 'rb') as f:
        data = pickle.load(f)
    
    frame_id = data['frame_id']
    graph = data['graph']
    positions = data['positions']
    
    if len(positions) < 3:
        continue
    
    # 凸包分解：分配垂直层级
    levels = assign_vertical_levels(positions, graph)
    
    # 分配水平位置
    h_positions = assign_horizontal_positions(positions, levels)
    
    # 推断阵型（新版：返回三种格式）
    formation_simple, formation_code, formation_detailed = infer_formation_from_levels(
        levels, graph, positions
    )
    
    # 转换层级和位置为字符串标签
    level_names = ['B', 'DM', 'M', 'AM', 'F']
    h_pos_names = ['L', 'LC', 'C', 'RC', 'R']
    
    vertical_levels_list = [level_names[l] for l in levels]
    horizontal_positions_list = [h_pos_names[h] for h in h_positions]
    
    # 保存结果（新增formation_code和formation_detailed）
    results.append({
        'frame_id': frame_id,
        'formation': formation_simple,          # 简化格式 (如'4-3-3')
        'formation_code': formation_code,        # Code格式 (如'4(003)3')
        'formation_detailed': formation_detailed, # 详细格式 (如'40033')
        'n_defenders': np.sum(levels == 0),
        'n_dm': np.sum(levels == 1),
        'n_midfielders': np.sum(levels == 2),
        'n_am': np.sum(levels == 3),
        'n_forwards': np.sum(levels == 4),
        'vertical_levels': vertical_levels_list,
        'horizontal_positions': horizontal_positions_list,
    })

# 转换为DataFrame
results_df = pd.DataFrame(results)

print(f"\n✅ 处理完成: {len(results_df)} 帧")
print(f"\n阵型分布（简化格式）:")
print(results_df['formation'].value_counts().head(10))
print(f"\n阵型分布（详细格式）:")
print(results_df['formation_detailed'].value_counts().head(10))

=== 开始批量处理 ===
处理 50084 个Shape Graph文件...



识别阵型: 100%|██████████| 50084/50084 [15:03<00:00, 55.43it/s]



✅ 处理完成: 50084 帧

阵型分布（简化格式）:
formation
3-5-2    15386
3-4-3     6762
4-4-2     5879
2-5-3     4534
2-4-4     3443
4-5-1     3108
2-6-2     2880
3-6-1     2832
5-4-1     1410
2-7-1      911
Name: count, dtype: int64

阵型分布（详细格式）:
formation_detailed
31132    6677
30133    4295
41122    4081
31222    3366
30142    3336
20134    2731
31123    2466
32122    2007
32221    1817
21133    1778
Name: count, dtype: int64


## 5. 时序平滑

In [5]:
def smooth_formations(formations, window_size=5):
    """
    使用滑动窗口众数平滑阵型序列
    
    参数:
        formations: 阵型序列
        window_size: 窗口大小
    
    返回:
        smoothed: 平滑后的阵型序列
    """
    smoothed = []
    for i in range(len(formations)):
        start = max(0, i - window_size // 2)
        end = min(len(formations), i + window_size // 2 + 1)
        window = formations[start:end]
        # 使用众数
        most_common = Counter(window).most_common(1)[0][0]
        smoothed.append(most_common)
    return smoothed

# 应用时序平滑（对三种格式分别平滑）
results_df['formation_smoothed'] = smooth_formations(results_df['formation'].tolist())
results_df['formation_code_smoothed'] = smooth_formations(results_df['formation_code'].tolist())
results_df['formation_detailed_smoothed'] = smooth_formations(results_df['formation_detailed'].tolist())

print("✅ 时序平滑完成")
print(f"\n平滑后阵型分布（简化格式）:")
print(results_df['formation_smoothed'].value_counts().head(10))
print(f"\n平滑后阵型分布（详细格式）:")
print(results_df['formation_detailed_smoothed'].value_counts().head(10))

✅ 时序平滑完成

平滑后阵型分布（简化格式）:
formation_smoothed
3-5-2    15182
3-4-3     6861
4-4-2     5841
2-5-3     4668
2-4-4     3445
4-5-1     3062
2-6-2     2923
3-6-1     2854
5-4-1     1298
2-7-1      942
Name: count, dtype: int64

平滑后阵型分布（详细格式）:
formation_detailed_smoothed
31132    6407
30133    4326
41122    4123
31222    3389
30142    3297
20134    2727
31123    2541
32122    2069
21133    1831
32221    1803
Name: count, dtype: int64


## 6. 保存结果

In [6]:
# 保存完整结果
output_file = OUTPUT_DIR / f'shapegraphs_baseline_results_{GAME_ID}.parquet'
results_df.to_parquet(output_file)
print(f"✅ 完整结果已保存: {output_file}")

# 保存阵型序列（用于对比评估）
formation_sequence = results_df[['frame_id', 'formation_smoothed']].copy()
formation_sequence.columns = ['frame_id', 'formation']
sequence_file = OUTPUT_DIR / f'shapegraphs_formation_sequence_{GAME_ID}.parquet'
formation_sequence.to_parquet(sequence_file)
print(f"✅ 阵型序列已保存: {sequence_file}")

# 保存CSV版本（便于查看）
csv_file = OUTPUT_DIR / f'shapegraphs_baseline_results_{GAME_ID}.csv'
results_df.to_csv(csv_file, index=False)
print(f"✅ CSV结果已保存: {csv_file}")

print(f"\n=== 批量处理完成 ===")
print(f"总帧数: {len(results_df)}")
print(f"主要阵型: {results_df['formation_smoothed'].mode()[0]}")

✅ 完整结果已保存: ..\..\..\data\morph_test\shapegraphs_baseline\shapegraphs_baseline_results_10517.parquet
✅ 阵型序列已保存: ..\..\..\data\morph_test\shapegraphs_baseline\shapegraphs_formation_sequence_10517.parquet
✅ CSV结果已保存: ..\..\..\data\morph_test\shapegraphs_baseline\shapegraphs_baseline_results_10517.csv

=== 批量处理完成 ===
总帧数: 50084
主要阵型: 3-5-2


## 总结

本notebook完成了以下工作：

1. ✅ 加载了所有Shape Graph文件
2. ✅ 批量处理进行阵型识别
3. ✅ 应用了时序平滑
4. ✅ 保存了3个输出文件：
   - `shapegraphs_baseline_results_{GAME_ID}.parquet` - 完整结果
   - `shapegraphs_formation_sequence_{GAME_ID}.parquet` - 阵型序列
   - `shapegraphs_baseline_results_{GAME_ID}.csv` - CSV版本

下一步：运行3.1.2.3进行可视化分析